In [20]:
# generated TLE format inspection

L = 69  # ожидаемая длина строки TLE
generated1 = '1 30001U 23654Y   23069.0.298980  0.00012642  35430-5 -66765-3 0 22703'

# https://ru.wikipedia.org/wiki/TLE
wiki_sample = '2 25544  51.6416 247.4627 0006703 130.5360 325.0288 15.72125391563537'
generated2  = '2 30001  96.1412 298.8961 0028500 301.5185 331.9435 12.86203311482223'
assert len(generated2) == 69, f"{generated2=} {len(generated2)=}"

def tle2(satnum: int) -> str:
    # result = (
    #     f"2 {satnum:05d}  {random.uniform(0, 180):8.4f}  "
    #     + f"{random.uniform(0, 360):8.4f}  {random.randint(0, 9999999):07d}  "
    #     + f"{random.uniform(0, 360):8.4f}  {random.uniform(0, 360):8.4f}  "
    #     + f"{random.uniform(12, 16):11.8f} {random.randint(1, 99999):5d}"
    # ).strip()

    norad_satnum = 25544  # 03–07            Satellite Catalog number
    inclination_degrees = 51.6416  # 09–16   Inclination (degrees)
    right_ascension = 247.4627  # 18–25      Right ascension of the ascending node (degrees),
    # eccentricity = 0006703  # 27–33 	     Eccentricity (unitless, decimal point assumed)
    eccentricity = 6703  # 27–33 	         Eccentricity (unitless, decimal point assumed)
    argument_of_perigee = 130.5360  # 35–42  Argument of perigee (degrees)
    mean_anomaly = 325.0288    # 44–51 	     Mean anomaly (degrees)
    mean_motion = 15.72125391  # 53–63 	     Mean motion (revolutions per day)
    revolution_number = 56353  # 64–68 	     Revolution number at epoch (revolutions)
    checksum = 7               # 69          Checksum (modulo 10)
    
    result = (
        f"2 {norad_satnum:05d} {inclination_degrees:8.4f} "
        + f"{right_ascension:8.4f} {eccentricity:07d} "
        + f"{argument_of_perigee:8.4f} {mean_anomaly:8.4f} {mean_motion:8.4f}"
        + f"{revolution_number:11.8f}{checksum:5d}"
    ).strip()

    # FIXME: верифицировать формат
    # assert len(result) == 69, f"{result=} {len(result)=}"
    return result

print(wiki_sample)
print(tle2(1234))

w = '''\
1 08820U 76039A   25295.13656333  .00000012  00000+0  00000+0 0  9997
2 08820 109.8410  97.8124 0044669 353.2367 191.6621  6.38664886897939'''
e = '''\
1 30000U 25688G   25149.77859350  0.00054814  17387-5  82091-3 0 75895
1 30000U 24358Q   25255.0.322882  0.00074281  51233-5  80033-3 0 26457
2 30000 161.0815 249.0919 0220141  28.2684  72.9240 14.06780130621037'''

print(f"{len('1 30000U 25688G   25149.77859350  0')=}")

assert len(e.split('\n')[0]) == L, f"{len(e.split('\n')[0])=}"


2 25544  51.6416 247.4627 0006703 130.5360 325.0288 15.72125391563537
2 25544  51.6416 247.4627 0006703 130.5360 325.0288  15.721356353.00000000    7
len('1 30000U 25688G   25149.77859350  0')=35


AssertionError: len(e.split('
')[0])=70

In [ ]:

# https://claude.ai/chat/603bb095-331d-4419-8d34-f957aa06a2df
# сгенерировано нейросетью claude
class TLEProvider(faker.providers.BaseProvider):
    """Генератор фейковых TLE (элементов описания орбиты)."""

    @staticmethod
    def _calculate_checksum(line: str) -> int:
        """Вычисляет контрольную сумму для строки TLE.

        Суммируются все цифры, минус добавляет 1, результат по модулю 10.
        """
        checksum = 0
        for char in line:
            if char.isdigit():
                checksum += int(char)
            elif char == '-':
                checksum += 1
        return checksum % 10

    @staticmethod
    def _format_exponential(value: float, width: int) -> str:
        """Форматирует число в научной нотации для TLE.

        Например: -0.11606E-4 → -11606-4
        Предполагается ведущая десятичная точка.
        """
        if value == 0:
            return '0' * (width - 2) + '+0'

        sign = '-' if value < 0 else ' '
        abs_value = abs(value)

        # Преобразуем в научную нотацию
        exponent = 0
        mantissa = abs_value

        if mantissa != 0:
            while mantissa >= 1.0:
                mantissa /= 10.0
                exponent += 1
            while mantissa < 0.1:
                mantissa *= 10.0
                exponent -= 1

        # Убираем ведущий 0.
        mantissa_str = f"{mantissa:.5f}"[2:2 + (width - 2)]  # Убираем "0."
        exp_sign = '+' if exponent >= 0 else '-'

        return f"{sign}{mantissa_str}{exp_sign}{abs(exponent)}"

    @classmethod
    def tle1(cls, satnum: int, launch_date: datetime.datetime) -> str:
        """Генерирует строку 1 TLE.

        Основано на спецификации:
        - Поле 1: Номер строки (1)
        - Поле 2: Каталожный номер спутника (5 цифр)
        - Поле 3: Классификация (U/C/S)
        - Поля 4-6: Международный идентификатор
        - Поля 7-8: Эпоха
        - Поле 9: Первая производная среднего движения
        - Поле 10: Вторая производная среднего движения
        - Поле 11: B* (коэффициент сопротивления)
        - Поле 12: Тип эфемерид (всегда 0)
        - Поле 13: Номер набора элементов
        - Поле 14: Контрольная сумма
        """
        # Поле 2: Satellite catalog number (колонки 3-7)
        satnum_str = f"{satnum:05d}"

        # Поле 3: Classification (колонка 8)
        classification = 'U'

        # Поля 4-6: International Designator (колонки 10-17)
        launch_year = launch_date.year % 100  # последние 2 цифры
        launch_num = random.randint(1, 999)  # номер запуска в году
        launch_piece = random.choice('ABCDEFGHIJKLMNOPQRSTUVWXYZ')  # часть запуска
        intl_desig = f"{launch_year:02d}{launch_num:03d}{launch_piece}  "

        # Поля 7-8: Epoch (колонки 19-32)
        epoch_year = random.randint(20, 25)  # 2020-2025
        day_of_year = random.randint(1, 365)
        fraction = random.random()
        epoch_day = f"{day_of_year:03d}.{fraction:.8f}"[:12]  # день.дробная часть
        epoch_str = f"{epoch_year:02d}{epoch_day}"

        # Поле 9: Первая производная среднего движения (колонки 34-43)
        first_deriv = random.uniform(-0.001, 0.001)
        first_deriv_str = f"{first_deriv: .8f}"

        # Поле 10: Вторая производная (колонки 45-52)
        second_deriv = random.uniform(-0.00001, 0.00001)
        second_deriv_str = cls._format_exponential(second_deriv, 8)

        # Поле 11: B* drag term (колонки 54-61)
        bstar = random.uniform(-0.001, 0.001)
        bstar_str = cls._format_exponential(bstar, 8)

        # Поле 12: Ephemeris type (колонка 63)
        ephemeris_type = '0'

        # Поле 13: Element set number (колонки 65-68)
        element_num = random.randint(0, 9999)
        element_num_str = f"{element_num:4d}"

        # Собираем строку без контрольной суммы
        line = (
            f"1 {satnum_str}{classification} {intl_desig} "
            f"{epoch_str} {first_deriv_str} {second_deriv_str} "
            f"{bstar_str} {ephemeris_type} {element_num_str}"
        )

        # Поле 14: Checksum (колонка 69)
        checksum = cls._calculate_checksum(line)
        line += f"{checksum}"

        return line

    @classmethod
    def tle2(cls, satnum: int) -> str:
        """Генерирует строку 2 TLE.

        Основано на спецификации:
        - Поле 1: Номер строки (2)
        - Поле 2: Каталожный номер спутника (5 цифр)
        - Поле 3: Наклонение (градусы)
        - Поле 4: Прямое восхождение узла (градусы)
        - Поле 5: Эксцентриситет (без десятичной точки)
        - Поле 6: Аргумент перигея (градусы)
        - Поле 7: Средняя аномалия (градусы)
        - Поле 8: Среднее движение (обороты/день)
        - Поле 9: Номер оборота на эпохе
        - Поле 10: Контрольная сумма
        """
        # Поле 2: Satellite catalog number (колонки 3-7)
        satnum_str = f"{satnum:05d}"

        # Поле 3: Inclination (колонки 9-16)
        inclination = random.uniform(0, 180)
        inclination_str = f"{inclination:8.4f}"

        # Поле 4: Right Ascension (колонки 18-25)
        raan = random.uniform(0, 360)
        raan_str = f"{raan:8.4f}"

        # Поле 5: Eccentricity (колонки 27-33)
        # Десятичная точка предполагается, т.е. 0.0006703 → 0006703
        eccentricity = random.uniform(0.0000001, 0.05)  # типичные значения для спутников
        eccentricity_int = int(eccentricity * 10000000)  # преобразуем в целое
        eccentricity_str = f"{eccentricity_int:07d}"

        # Поле 6: Argument of Perigee (колонки 35-42)
        arg_perigee = random.uniform(0, 360)
        arg_perigee_str = f"{arg_perigee:8.4f}"

        # Поле 7: Mean Anomaly (колонки 44-51)
        mean_anomaly = random.uniform(0, 360)
        mean_anomaly_str = f"{mean_anomaly:8.4f}"

        # Поле 8: Mean Motion (колонки 53-63)
        # Для низких орбит обычно 12-16 оборотов/день
        mean_motion = random.uniform(12.0, 16.0)
        mean_motion_str = f"{mean_motion:11.8f}"

        # Поле 9: Revolution number (колонки 64-68)
        rev_number = random.randint(1, 99999)
        rev_number_str = f"{rev_number:5d}"

        # Собираем строку без контрольной суммы
        line = (
            f"2 {satnum_str} {inclination_str} {raan_str} "
            f"{eccentricity_str} {arg_perigee_str} {mean_anomaly_str} "
            f"{mean_motion_str}{rev_number_str}"
        )

        # Поле 10: Checksum (колонка 69)
        checksum = cls._calculate_checksum(line)
        line += f"{checksum}"

        return linth e